# Diffusion Sources: Colab GPU Training
Drive stores datasets and checkpoints; GitHub provides versioned code.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/diffusion-sources'
REPOSITORY = 'https://github.com/1habibi/diffusion-sources-localization-.git'


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))


In [ ]:
!rm -rf /content/diffusion-sources
!git clone {REPOSITORY} /content/diffusion-sources
%cd /content/diffusion-sources
!pip install -q networkx numpy matplotlib PyYAML scikit-learn streamlit tqdm torch-geometric
!pip install -q -e . --no-deps
!git rev-parse HEAD


In [ ]:
from pathlib import Path
import yaml
base = yaml.safe_load(Path('configs/train_facebook.yaml').read_text())
base['data']['directory'] = f'{DRIVE_ROOT}/data/facebook_main'
base['training']['device'] = 'cuda'
base['training']['seed'] = 7026
base['training']['resume'] = True
Path('/content/train_joint.yaml').write_text(yaml.safe_dump(base, sort_keys=False))
print(yaml.safe_dump(base, sort_keys=False))


In [ ]:
OUTPUT = f'{DRIVE_ROOT}/reports/joint_full/seed_7026'
!python scripts/train_model.py --config /content/train_joint.yaml --output {OUTPUT}


In [ ]:
import json
metrics = json.loads(Path(OUTPUT, 'metrics.json').read_text())
print(json.dumps(metrics['prediction_metrics']['joint_estimated_k']['all'], indent=2))
print(json.dumps(metrics['runtime'], indent=2))
